# BNPL Default Risk: Batch Inference

## Purpose
Verify that batch inference on a CSV works end to end. The script path is `src/inference.py --csv <file>`. This notebook builds a small test CSV from the raw data, runs the batch path, and inspects the output.

## Workflow
1. Build a test CSV from the first rows of the raw dataset
2. Run `inference.py --csv` through subprocess
3. Inspect the output CSV, check that the probability columns sum to 1
4. Try two edge cases: an unseen category and a missing required column
5. Clean up the test files so they do not get committed

### Imports and Configuration

In [1]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from config import RAW_DATA_PATH, TARGET_COL

### Build a Test CSV
Take the first 20 rows of the raw dataset, drop the target column, and save. This gives a valid input with real values and the correct dtypes.

In [2]:
df_raw = pd.read_csv(RAW_DATA_PATH)
test_csv = PROJECT_ROOT / "test_batch.csv"
df_raw.drop(columns=[TARGET_COL]).head(20).to_csv(test_csv, index=False)
print(f"Wrote 20 rows to {test_csv}")
print(f"Columns: {list(df_raw.drop(columns=[TARGET_COL]).columns)}")

Wrote 20 rows to c:\Users\Alexz\BNPL_ML\test_batch.csv
Columns: ['Customer_ID', 'Age', 'Employment_Status', 'Income_USD', 'Credit_Score', 'Total_BNPL_Active_Loans', 'Total_BNPL_Debt_USD', 'Late_Payment_History', 'Shopping_Category_Most_Frequent', 'Average_Transaction_Value_USD']


### Run Batch Inference
Call the inference script through subprocess so the notebook mirrors the actual CLI usage.

In [3]:
result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "src" / "inference.py"), "--csv", str(test_csv)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

Row 6: Credit_Score missing, using training median.
Row 10: Income_USD missing, using training median.
Saved: c:\Users\Alexz\BNPL_ML\test_batch_with_predictions.csv



### Inspect the Output
The script writes a sibling file with `_with_predictions` appended to the input name. The output keeps the original columns and adds `Predicted_Risk`, `Prob_Low`, `Prob_Medium`, and `Prob_High`.

In [4]:
out_csv = PROJECT_ROOT / "test_batch_with_predictions.csv"
df_out = pd.read_csv(out_csv)
print(f"Shape: {df_out.shape}")
df_out[["Age", "Income_USD", "Credit_Score", "Predicted_Risk",
        "Prob_Low", "Prob_Medium", "Prob_High"]].head()

Shape: (20, 14)


,Age,Income_USD,Credit_Score,Predicted_Risk,Prob_Low,Prob_Medium,Prob_High
0,29,73405.0,637.0,Low,0.999465,0.000151,0.000383
1,64,53732.0,815.0,Low,0.999740,0.000188,0.000073
2,26,69930.0,723.0,Low,0.999448,0.000259,0.000293
3,19,96941.0,716.0,Low,0.999527,0.000288,0.000185
4,55,15959.0,675.0,Low,0.997820,0.001776,0.000404


### Verifying Probabilities Sum to One

In [5]:
prob_cols = ["Prob_Low", "Prob_Medium", "Prob_High"]
row_sums = df_out[prob_cols].sum(axis=1)
print(f"Min row sum: {row_sums.min():.6f}")
print(f"Max row sum: {row_sums.max():.6f}")
print(f"All within 1e-4: {((row_sums - 1).abs() < 1e-4).all()}")

Min row sum: 1.000000
Max row sum: 1.000000
All within 1e-4: True


### Edge Case: Unseen Category
Replace one row's `Employment_Status` with a value the model has never seen. The inference code maps unknown categories to code 0, which is reserved for that purpose, so the run should complete without crashing.

In [6]:
df_edge = df_raw.drop(columns=[TARGET_COL]).head(5).copy()
df_edge.loc[0, "Employment_Status"] = "Astronaut"
edge_csv = PROJECT_ROOT / "test_batch_edge.csv"
df_edge.to_csv(edge_csv, index=False)

result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "src" / "inference.py"), "--csv", str(edge_csv)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

edge_out = pd.read_csv(PROJECT_ROOT / "test_batch_edge_with_predictions.csv")
edge_out[["Employment_Status", "Predicted_Risk"]]

Saved: c:\Users\Alexz\BNPL_ML\test_batch_edge_with_predictions.csv

c:\Users\Alexz\BNPL_ML\src\utils.py:297: UserWarning: Unknown category 'Astronaut' for Employment_Status; using code 0
  warnings.warn(f"Unknown category '{val}' for {col}; using code 0")



,Employment_Status,Predicted_Risk
0,Astronaut,Low
1,Employed,Low
2,Freelancer,Low
3,Employed,Low
4,Student,Low


### Edge Case: Missing Column
Drop `Credit_Score` and confirm the script fails cleanly with a message rather than producing a broken output.

In [7]:
df_missing = df_raw.drop(columns=[TARGET_COL, "Credit_Score"]).head(5)
missing_csv = PROJECT_ROOT / "test_batch_missing.csv"
df_missing.to_csv(missing_csv, index=False)

result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "src" / "inference.py"), "--csv", str(missing_csv)],
    capture_output=True, text=True,
)
print(result.stdout)
print(f"Output file created: {(PROJECT_ROOT / 'test_batch_missing_with_predictions.csv').exists()}")

Missing columns: ['Credit_Score']

Output file created: False


### Cleanup
The `.gitignore` already blocks `test_batch*.csv`, so these files cannot be committed by accident. This cell removes them from the working directory as well, so the repo stays clean after a run.

In [8]:
for name in [
    "test_batch.csv",
    "test_batch_with_predictions.csv",
    "test_batch_edge.csv",
    "test_batch_edge_with_predictions.csv",
    "test_batch_missing.csv",
    "test_batch_missing_with_predictions.csv",
]:
    p = PROJECT_ROOT / name
    if p.exists():
        p.unlink()
        print(f"Removed {name}")

Removed test_batch.csv
Removed test_batch_with_predictions.csv
Removed test_batch_edge.csv
Removed test_batch_edge_with_predictions.csv
Removed test_batch_missing.csv
